
---

# What is CatBoost?

**CatBoost (Categorical Boosting)** is a gradient boosting algorithm developed by Yandex.

It is specially designed to:

* Handle categorical features natively
* Reduce prediction shift
* Reduce overfitting in boosting

---

# 1️⃣ Core Optimization Problem

Like XGBoost and LightGBM, CatBoost minimizes:

[
\mathcal{L} =
\sum_{i=1}^{n}
l(y_i, \hat{y}*i)
+
\sum*{t=1}^{T}
\Omega(f_t)
]

Model form:

[
\hat{y}*i = \sum*{t=1}^{T} f_t(x_i)
]

And trees are added sequentially.

So at high level → same boosting framework.

---

# 2️⃣ The Main Innovation: Ordered Boosting

### Problem in Standard Gradient Boosting

When computing gradient:

[
g_i = \frac{\partial l(y_i, \hat{y}_i)}{\partial \hat{y}_i}
]

The prediction ( \hat{y}_i ) was trained using the same data point.

This causes **prediction shift**.

Mathematically:

[
E[g_i] \neq \text{true gradient}
]

Because the model has already seen ( y_i ).

---

## CatBoost Solution: Ordered Boosting

Instead of training using full dataset at once:

1. Create random permutation of data.
2. For each sample ( i ), compute prediction using only previous samples in permutation.

Formally:

For permutation ( \pi ):

[
\hat{y}*i^{(t)} =
\sum*{k < i}
f_t(x_{\pi_k})
]

Thus gradient for sample i:

[
g_i =
\partial_{\hat{y}}
l(y_i, \hat{y}_i^{(t-1)})
]

is computed without seeing its own label during tree building.

This makes gradient unbiased.

---

# 3️⃣ Handling Categorical Features (Major Strength)

Traditional methods:

* One-hot encoding
* Target encoding (leaks data)

CatBoost introduces **Ordered Target Statistics**.

For category value ( c ):

Instead of:

[
\frac{\sum y_i}{count(c)}
]

It computes:

[
\frac{
\sum_{j < i, x_j = c} y_j
+
a \cdot p
}
{
\sum_{j < i, x_j = c} 1
+
a
}
]

Where:

* Only earlier samples in permutation used
* ( a ) = smoothing parameter
* ( p ) = prior

This avoids target leakage.

---

# 4️⃣ Tree Structure: Oblivious Trees

Unlike XGBoost & LightGBM, CatBoost uses:

## Symmetric (Oblivious) Trees

At each depth, the same split is applied to all nodes.

Example:

Depth 1: Split on Feature A
Depth 2: Split on Feature B

So every path uses same splitting sequence.

Mathematically:

Tree defines:

[
f(x) = w_{b(x)}
]

Where:

[
b(x) \in {0,1}^d
]

Binary path of length d.

Number of leaves:

[
2^d
]

---

## Why Oblivious Trees?

1. Faster prediction
2. Regularization effect
3. Better GPU performance
4. Lower variance

---

# 5️⃣ Objective Approximation

Like XGBoost, CatBoost uses:

Second-order Taylor expansion:

[
l(y_i, \hat{y}_i + f(x_i))
\approx
l + g_i f(x_i)

* \frac{1}{2} h_i f^2(x_i)
  ]

And optimal leaf weight:

[
w_j^* =
-------

\frac{\sum g_i}
{\sum h_i + \lambda}
]

So core math identical to XGBoost.

Difference lies in:

* Gradient computation (ordered)
* Tree structure (symmetric)

---

# 6️⃣ Regularization in CatBoost

CatBoost controls overfitting via:

* Ordered boosting
* Symmetric trees
* L2 leaf regularization
* Depth constraint
* Bayesian bootstrapping

Bayesian bootstrap weight:

[
w_i \sim \text{Exponential}(1)
]

Instead of uniform sampling.

---

# 7️⃣ Mathematical Comparison: XGBoost vs LightGBM vs CatBoost

| Feature              | XGBoost           | LightGBM          | CatBoost                |
| -------------------- | ----------------- | ----------------- | ----------------------- |
| Boosting type        | Gradient boosting | Gradient boosting | Ordered boosting        |
| Tree growth          | Level-wise        | Leaf-wise         | Symmetric               |
| Categorical handling | Manual encoding   | Manual encoding   | Native ordered encoding |
| Gradient bias        | Yes               | Yes               | No                      |
| Split strategy       | Gain-based        | Gain-based        | Gain-based              |
| Overfitting risk     | Medium            | Higher            | Lower                   |
| Speed                | Fast              | Very fast         | Medium                  |

---

# 8️⃣ Mathematical View of Overfitting Control

### XGBoost:

[
\Omega(f) = \gamma T + \frac{1}{2} \lambda \sum w_j^2
]

Regularization explicit.

---

### LightGBM:

Relies more on:

* Leaf constraint
* GOSS sampling

---

### CatBoost:

Reduces overfitting via:

[
E[\hat{y}_i] \text{ unbiased}
]

Because ordered gradient reduces shift.

This is statistically more principled.

---

# 9️⃣ Computational Complexity

Let:

* n = samples
* d = depth
* T = trees

CatBoost symmetric tree:

[
O(T \cdot n \cdot d)
]

Efficient because:

* Fixed structure
* Fast bitwise operations

---

# 🔟 When to Use CatBoost

Best when:

✔ Many categorical variables
✔ Small-medium dataset
✔ Want minimal preprocessing
✔ Avoid target leakage
✔ Need stable model

Not ideal when:

✖ Extremely large dataset (LightGBM faster)
✖ No categorical features

---

# 1️⃣1️⃣ Conceptual Summary

All three minimize similar objective:

[
\sum l(y_i, \hat{y}_i) + \Omega(f)
]

But differ in:

| Algorithm | Key Innovation                     |
| --------- | ---------------------------------- |
| XGBoost   | Regularized second-order boosting  |
| LightGBM  | Leaf-wise fast growth              |
| CatBoost  | Ordered boosting + symmetric trees |

---

# Final Exam-Ready Definition

> CatBoost is a gradient boosting algorithm that uses ordered boosting and symmetric decision trees to eliminate prediction shift and handle categorical variables without target leakage.

---
